# Extension Experiment A — Dual-Confidence Selection Baseline (base paper's final method)
Evaluates the conference paper's final mechanism — run BOTH classifiers on every image and keep the higher-confidence prediction — on our benchmarks, seeds, and corruption conditions, using the plain EfficientNet-B0 and ConvNeXt-Tiny checkpoints already trained. Cost is fixed at 0.42 + 4.47 = 4.89 GMACs per image (both models always run).

This row does two jobs in the extended paper: it restores direct methodological continuity with the base work, and it quantifies what dual execution costs relative to routed execution (ARC-D) and to simply always running the heavy model.

**Eval-only. No training. Needs: GPU runtime + Drive with `arcd_final/final_ckpts/`. ~15–20 min.**

In [ ]:
!pip -q install timm
import os, random, csv
import numpy as np, torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import cv2, timm
device="cuda" if torch.cuda.is_available() else "cpu"
assert device=="cuda", "use a GPU runtime"
from google.colab import drive; drive.mount("/content/drive")
BASE="/content/drive/MyDrive/arcd_final"; CKPT=os.path.join(BASE,"final_ckpts")
G_CHEAP, G_HEAVY = 0.398, 4.470
print(sorted(os.listdir(CKPT)))

In [ ]:
import urllib.request, subprocess
def load_test(ds):
    if ds=="gsd":
        if not os.path.exists("GroceryStoreDataset"):
            subprocess.run(["git","clone","-q","https://github.com/marcusklasson/GroceryStoreDataset.git"],check=True)
        R="GroceryStoreDataset/dataset"; out=[]
        with open(os.path.join(R,"test.txt")) as f:
            for line in f:
                line=line.strip()
                if not line: continue
                p=[x.strip() for x in line.split(",")]
                out.append((os.path.join(R,p[0]),int(p[1])))
        return out, 81
    if not os.path.exists("images"):
        url="http://aisdatasets.informatik.uni-freiburg.de/freiburg_groceries_dataset/freiburg_groceries_dataset.tar.gz"
        print("downloading (~1GB)..."); urllib.request.urlretrieve(url,"fg.tar.gz")
        subprocess.run(["tar","-xf","fg.tar.gz"],check=True)
    RAW="https://raw.githubusercontent.com/PhilJd/freiburg_groceries_dataset/master/splits"
    if not os.path.exists("test0.txt"): urllib.request.urlretrieve(f"{RAW}/test0.txt","test0.txt")
    seen=set(); out=[]
    with open("test0.txt") as fh:
        for line in fh:
            line=line.strip()
            if not line: continue
            rel,lab=line.rsplit(" ",1)
            if rel in seen: continue
            seen.add(rel); out.append((os.path.join("images",rel),int(lab)))
    return out, 25

MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
def to_t(img):
    t=torch.from_numpy(img).float().permute(2,0,1)/255.0
    for c in range(3): t[c]=(t[c]-MEAN[c])/STD[c]
    return t
def corrupt(img,kind,sev,idx=0):
    if kind=="none" or sev==0: return img
    if kind=="blur":
        k=[3,5,7,11,15][sev-1]; return cv2.GaussianBlur(img,(k,k),0)
    if kind=="jpeg":
        q=[40,30,20,12,7][sev-1]
        _,e=cv2.imencode(".jpg",cv2.cvtColor(img,cv2.COLOR_RGB2BGR),[cv2.IMWRITE_JPEG_QUALITY,q])
        return cv2.cvtColor(cv2.imdecode(e,1),cv2.COLOR_BGR2RGB)
    if kind=="occlusion":
        s=[0.1,0.2,0.3,0.4,0.5][sev-1]; side=int(224*s)
        r=random.Random(10_000*sev+idx)
        x0=r.randint(0,224-side); y0=r.randint(0,224-side)
        img=img.copy(); img[y0:y0+side,x0:x0+side]=127; return img
class TDS(Dataset):
    def __init__(self,items,kind,sev): self.items,self.kind,self.sev=items,kind,sev
    def __len__(self): return len(self.items)
    def __getitem__(self,i):
        p,l=self.items[i]
        img=np.array(Image.open(p).convert("RGB").resize((224,224)))
        return to_t(corrupt(img,self.kind,self.sev,idx=i)), l

In [ ]:
@torch.no_grad()
def conf_correct(model,items,kind,sev,bs=64):
    model.eval()
    dl=DataLoader(TDS(items,kind,sev),batch_size=bs,num_workers=2,pin_memory=True)
    C,K=[],[]
    for x,y in dl:
        p=torch.softmax(model(x.to(device)),1); c,pred=p.max(1)
        C.append(c.cpu().numpy()); K.append((pred.cpu()==y).numpy())
    return np.concatenate(C),np.concatenate(K)

def boot(diff,n=5000):
    idx=np.random.randint(0,len(diff),size=(n,len(diff)))
    b=diff[idx].mean(axis=1); return diff.mean(),np.percentile(b,5),np.percentile(b,95)

CONDS=[("clean","none",0),("jpeg_s2","jpeg",2),("blur_s2","blur",2),("occ_s1","occlusion",1)]
G_DUAL=G_CHEAP+G_HEAVY
rows=[]
for ds in ("gsd","freiburg"):
    items,NC=load_test(ds)
    agg={t:[] for t,_,_ in CONDS}; pooled={t:{"dual":[],"heavy":[]} for t,_,_ in CONDS}
    for seed in (0,1,2):
        c=timm.create_model("efficientnet_b0",pretrained=False,num_classes=NC).to(device)
        c.load_state_dict(torch.load(os.path.join(CKPT,f"{ds}_cheap_seed{seed}.pt"),map_location=device))
        h=timm.create_model("convnext_tiny",pretrained=False,num_classes=NC).to(device)
        h.load_state_dict(torch.load(os.path.join(CKPT,f"{ds}_heavy_seed{seed}.pt"),map_location=device))
        for tag,kind,sev in CONDS:
            cc_,kc=conf_correct(c,items,kind,sev)
            ch_,kh=conf_correct(h,items,kind,sev)
            kdual=np.where(cc_>=ch_,kc,kh)           # base paper's final method
            agg[tag].append(kdual.mean())
            pooled[tag]["dual"].append(kdual.astype(float))
            pooled[tag]["heavy"].append(kh.astype(float))
            rows.append(dict(dataset=ds,seed=seed,system="Dual-confidence",condition=tag,
                             acc=kdual.mean(),gmacs=G_DUAL))
        del c,h; torch.cuda.empty_cache()
        print(f"{ds} seed {seed} done")
    print(f"\n===== {ds}: Dual-confidence [base paper method], {G_DUAL:.2f} GMACs =====")
    for tag,_,_ in CONDS:
        a=np.array(agg[tag])*100
        d=np.mean(pooled[tag]["dual"],axis=0)-np.mean(pooled[tag]["heavy"],axis=0)
        m,lo,hi=boot(d)
        print(f"  {tag:8s} acc={a.mean():5.1f}±{a.std():3.1f}%   vs heavy: {m*100:+.2f}% [{lo*100:+.2f},{hi*100:+.2f}]")

fn=os.path.join(BASE,"dual_confidence_results.csv")
with open(fn,"w",newline="") as f:
    w=csv.DictWriter(f,fieldnames=["dataset","seed","system","condition","acc","gmacs"])
    w.writeheader()
    for r in rows: w.writerow(r)
print("\nsaved ->",fn)

Paste back the two summary blocks. Expected shape of the result: dual-confidence lands at or fractionally above always-heavy accuracy (the ensemble effect is small when one model dominates) at 4.89 GMACs — i.e. **more expensive than simply running the heavy model** and 2.6–4.4× the cost of ARC-D. That single row is the empirical case for the redesign, stated in the extension as: the conference mechanism preserves accuracy by paying for both models; the extended design pays for the second model only when the first is unsure.